# s28 — Self-Assigning Workers

**What this teaches:** how to attach an *autonomous worker goroutine* to the shared `tasks` graph from [s12_tasks](../s12_tasks/s12_tasks.ipynb) so it claims and completes tasks without the LLM being involved on every step. The leader agent only watches progress via `task_list`.

**Concept anchor:** `tasks.Graph` is concurrency-safe — `ClaimNext` is a transactional dequeue, and `Update(id, StatusDone, ...)` is the matching commit. That is the entire contract: any number of workers (goroutines, separate processes, or even other agents) can drain the same queue safely. The LLM becomes a *watcher* rather than a *driver* of the work.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars before launching Jupyter, e.g.:
  ```
  export YOKE_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export YOKE_MODEL=qwen2.5-coder
  ```
  Defaults to a local Ollama. Swap in `anthropic` / `openai` / `gemini` for hosted providers — see [docs/providers.md](../../docs/providers.md).

## 1. Imports

Same toolkit as [s01_loop](../s01_loop/s01_loop.ipynb) plus the `internal/tasks` graph (which we already met in [s12_tasks](../s12_tasks/s12_tasks.ipynb)).

In [ ]:
import (
    "context"
    "fmt"
    "os"
    "time"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/stream"
    "github.com/blouargant/yoke/internal/tasks"
)

## 2. Helper

Notebook-safe `must` — panics instead of `os.Exit`.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Seed the shared task graph

`tasks.New("")` creates an unscoped in-memory graph (pass a `userID` for session isolation in real apps). We push three tasks at medium priority — the worker will see them in FIFO order within their priority band.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
g := tasks.New("")
for _, d := range []string{"build", "test", "release"} {
    _, _ = g.Create(d, nil, tasks.PriorityMedium)
}
fmt.Println("seeded 3 tasks")

## 4. Launch the worker goroutine

The worker loops forever: `ClaimNext("worker-1")` atomically takes a task and marks it in-progress (no two workers will ever get the same one). When the queue is empty it sleeps briefly. In a real system this loop would be the unit doing the actual build / test / release — here we just immediately `Update` to `StatusDone` to simulate fast completion.

In [ ]:
go func() {
    for {
        t, _ := g.ClaimNext("worker-1")
        if t == nil {
            time.Sleep(300 * time.Millisecond)
            continue
        }
        _, _ = g.Update(t.ID, tasks.StatusDone, "ok")
    }
}()

## 5. Build a watcher agent

The agent only needs the `tasks` tools (`task_list`, `task_update`, …) — it does not create or claim tasks here, just reports on them. We sleep briefly to give the worker a beat to drain the queue, then run a single turn that asks for a status report.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s28_self_assign",
    Description: "Watches a self-assigning task graph.",
    Instruction: "Call task_list to report progress.",
    Model:       llm,
    Tools:       g.Tools(),
})
must(err)
r, err := agentkit.Runner("s28", a)
must(err)

time.Sleep(500 * time.Millisecond)
prompt := "Report the current task graph status."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))
fmt.Println()

## What to look for

- By the time the LLM calls `task_list`, all three tasks should already be `done` — the worker drained them before the model woke up. The LLM is a passive observer of state it never produced.
- This is the same `tasks.Graph` you saw in [s12_tasks](../s12_tasks/s12_tasks.ipynb), just with an external producer/consumer alongside the agent. The graph's locking guarantees mean you can mix human-driven, agent-driven, and worker-driven mutations without races.
- Pair this with [s26_mailbox](../s26_mailbox/s26_mailbox.ipynb) and you have the building blocks for a small multi-agent pipeline: workers consume tasks, mailboxes route results.

## Try it yourself

1. Increase the worker's sleep to `2 * time.Second` so the model sees in-flight tasks (some `in_progress`, some `pending`) when it calls `task_list`.
2. Spawn two worker goroutines (`worker-1`, `worker-2`) and confirm via the final `task_list` output that no task was claimed twice — the `ClaimNext` invariant is what makes this safe.